In [1]:
import numpy as np
import pandas as pd
import holoviews as hv
import panel as pn
from scipy.linalg import eigh
from juliacall import Main

pn.extension("katex")
hv.extension("bokeh")
np.set_printoptions(precision=3, suppress=True)


In [2]:
def diagnostic_table(values, *, title="Diagnostics"):
    rows = []
    for name, value in values.items():
        if isinstance(value, (float, np.floating)):
            shown = f"{value:.3e}"
        elif isinstance(value, np.ndarray):
            shown = np.array2string(value, precision=4, suppress_small=True)
        else:
            shown = str(value)
        rows.append((name, shown))
    table = hv.Table(rows, kdims=["Quantity"], vdims=["Value"]).opts(
        width=600, height=max(120, 32 * (len(rows) + 1))
    )
    return pn.Column(f"#### {title}", table, sizing_mode="fixed")


In [3]:
Main.seval(r"""
using LinearAlgebra

function lapack_gsvd_factors(A_input, B_input)
    A = Matrix{Float64}(A_input)
    B = Matrix{Float64}(B_input)
    size(A, 2) == size(B, 2) || throw(ArgumentError("A and B need the same number of columns"))
    F = svd(A, B)
    size(F.R0, 1) == size(A, 2) || throw(ArgumentError("the stacked pair must have full column rank"))
    X = F.Q / F.R0
    return (
        U = Matrix(F.U), V = Matrix(F.V),
        C = Matrix(F.D1), S = Matrix(F.D2), X = Matrix(X),
        c = Vector(F.alpha), s = Vector(F.beta), k = F.k, l = F.l,
    )
end
""")


lapack_gsvd_factors (generic function with 1 method)

<div style="height:1.5cm;"><div style="width:100%;text-align:center;"><strong style="color:darkred;font-size:36px;">A GSVD Case Study: Two Chapin Pond Stations</strong></div></div>

# 1. Purpose and Question

This notebook applies the generalized singular value decomposition to matched water-quality measurements from two monitoring stations in Chapin Pond, Morris County, New Jersey.

The question is deliberately narrow: **which linear combinations of measured indicators have larger or smaller pooled-standardized signal energy at Station 1 relative to Station 2?** This is a matrix-analysis question, not a claim that one station has better water quality.

For a vector of indicator weights $x$, the pair of matrices maps $x$ to two time-series score vectors. The GSVD finds shared right directions that organize the ratio

$\qquad\displaystyle{h(x)=\frac{\lVert A x\rVert_2^2}{\lVert B x\rVert_2^2}.
}$

The computations shown use the tested LAPACK GSVD through Julia's `svd(A, B)`, as in [**GSVD_computations.ipynb**](GSVD_computations.ipynb) and [**GSVD_regularization.ipynb**](GSVD_regularization.ipynb).

**Why perform this analysis?** For an environmental engineer comparing two monitoring locations, the GSVD can identify combinations of indicators whose measured signal is concentrated more strongly at one location than the other. Those modes can help prioritize follow-up measurements and generate hypotheses about multivariate differences; they do not identify a cause or rank overall water quality.

**What could a larger study conclude?** With representative sampling across relevant seasons and conditions, a stable mode with $\gamma_i>1$ has greater pooled-centered score energy at Station 1; one with $\gamma_i<1$ has greater energy at Station 2. A large sample can make these directions and ratios more precisely estimated, but the GSVD itself gives no confidence interval or significance test. For an inferential claim, resample matched dates (or seasonal blocks), repeat preprocessing and the full factorization, and report uncertainty and mode stability. A repeatable difference can guide follow-up monitoring; it does not identify a cause or establish which station has better water quality.

Here, both matrices are centered at one pooled mean. Thus $\lVert Ax\rVert_2^2$ and $\lVert Bx\rVert_2^2$ include within-station temporal scatter **and** each station's mean offset from the pooled mean: this is a comparison of second-moment energy, not a variance-only comparison. Although dates are matched, the GSVD objective uses the two matrices' separate Gram matrices and does not use date-by-date differences. If the engineering question concerns variability alone, center each station separately or compare within-station covariance; analyze station mean offsets separately.

# 2. Monitoring Data and its Limits

The observations are public NJDEP lake-monitoring results distributed through the U.S. EPA Water Quality Portal.  
The two site identifiers are `NJDEP_BFBM-NJLM-0188-1` and `NJDEP_BFBM-NJLM-0188-2`.  
Each has routine lab samples on the same eight dates from May 2023 through October 2024.  
* [Station 1 record](https://www.waterqualitydata.us/provider/STORET/NJDEP_BFBM/NJDEP_BFBM-NJLM-0188-1/),  
* [Station 2 record](https://www.waterqualitydata.us/provider/STORET/NJDEP_BFBM/NJDEP_BFBM-NJLM-0188-2/), [NJDEP lake-monitoring program](https://dep.nj.gov/wms/bfbm/lake-monitoring/).

To keep the notebook reproducible without a network connection, the exact selected result values are embedded below.  
The snapshot was retrieved on 2026-10-05 from the [Water Quality Portal result service](https://www.waterqualitydata.us/webservices_documentation/).  
The direct [query for both stations](https://www.waterqualitydata.us/data/Result/search?siteid=NJDEP_BFBM-NJLM-0188-1%3BNJDEP_BFBM-NJLM-0188-2&mimeType=csv&zip=no&providers=STORET) returns raw result rows.  
Dates are matched sampling rounds, not a continuous time series.

In [4]:
dates = pd.to_datetime([
    "2023-05-02", "2023-07-06", "2023-09-19", "2023-11-14",
    "2024-03-21", "2024-05-22", "2024-07-24", "2024-10-15",
])

# Columns: chlorophyll a (ug/L), alkalinity (mg/L), chloride (mg/L),
# sulfate (mg/L), and total Kjeldahl nitrogen (mg/L).
A_raw = np.array([
    [5.12, 12.1, 116.0, 4.43, 0.319],
    [10.89, 22.0, 87.5, 3.13, 0.385],
    [44.83, 9.0, 43.6, 1.87, 0.540],
    [20.01, 23.0, 49.9, 4.42, 0.396],
    [13.81, 37.0, 196.0, 7.38, 0.340],
    [11.92, 44.0, 147.0, 6.86, 0.330],
    [54.68, 35.0, 131.0, 4.77, 0.520],
    [37.72, 21.0, 80.5, 6.30, 0.900],
])
B_raw = np.array([
    [4.22, 11.1, 90.5, 2.84, 0.303],
    [8.26, 22.0, 88.2, 2.84, 0.513],
    [45.71, 8.5, 41.2, 1.87, 0.607],
    [17.35, 21.0, 48.2, 4.02, 0.377],
    [13.88, 33.0, 196.0, 8.20, 0.210],
    [6.56, 50.0, 148.0, 6.70, 0.390],
    [60.99, 32.0, 135.0, 5.21, 0.490],
    [37.68, 22.0, 80.3, 5.95, 0.740],
])

features = ["Chlorophyll a", "Alkalinity", "Chloride", "Sulfate", "TKN"]
units = ["ug/L", "mg/L", "mg/L", "mg/L", "mg/L"]
raw_table = pd.concat([
    pd.DataFrame(A_raw, columns=features).assign(Date=dates, Station="1"),
    pd.DataFrame(B_raw, columns=features).assign(Date=dates, Station="2"),
], ignore_index=True)[["Date", "Station", *features]]
raw_table


,Date,Station,Chlorophyll a,Alkalinity,Chloride,Sulfate,TKN
0,2023-05-02,1,5.12,12.1,116.0,4.43,0.319
1,2023-07-06,1,10.89,22.0,87.5,3.13,0.385
2,2023-09-19,1,44.83,9.0,43.6,1.87,0.540
3,2023-11-14,1,20.01,23.0,49.9,4.42,0.396
4,2024-03-21,1,13.81,37.0,196.0,7.38,0.340
5,2024-05-22,1,11.92,44.0,147.0,6.86,0.330
6,2024-07-24,1,54.68,35.0,131.0,4.77,0.520
7,2024-10-15,1,37.72,21.0,80.5,6.30,0.900
8,2023-05-02,2,4.22,11.1,90.5,2.84,0.303
9,2023-07-06,2,8.26,22.0,88.2,2.84,0.513


The five selected analytes are present at both stations on all eight dates with matching units and sample fractions:  
total chlorophyll *a*, total alkalinity, total chloride, total sulfate, and unfiltered total Kjeldahl nitrogen.  
The table omits three tempting alternatives for data-quality reasons:

- **Total phosphorus** changes from unfiltered to total sample fraction in 2024, so the series is not definitionally consistent across dates.
- **Nitrate plus nitrite** includes results at or near the reported quantitation limit; treating those values as precise measurements would overstate their information.
- **Total suspended solids** is missing for two of the eight matched dates.

Equipment-blank quality-control records are not environmental observations and are excluded.  
This selection is a transparent teaching choice, not a complete assessment of the pond.

In [5]:
raw_display = raw_table.copy()
raw_display[features] = raw_display[features].round(3)
pn.Column( "## Raw Data", pn.pane.DataFrame(raw_display, height=360))

Column
    [0] Markdown(str)
    [1] DataFrame(DataFrame, height=360)

## 2.1 First Look at the Data

The variables use different units and scales. Before comparing linear combinations, plot each measurement after centering and scaling  
with **one shared mean and scale per analyte**, computed from the pooled values at both stations.  
Using common transformations is essential: separate station-wise standardization would erase some of the between-station scale differences that the comparison is meant to retain.

In [6]:
pooled = np.vstack((A_raw, B_raw))
feature_mean = pooled.mean(axis=0)
feature_scale = pooled.std(axis=0, ddof=1)
if np.any(feature_scale == 0):
    raise ValueError("Every selected feature must vary in the pooled data")

A = (A_raw - feature_mean) / feature_scale
B = (B_raw - feature_mean) / feature_scale

feature_info = pd.DataFrame({
    "Indicator": features,
    "Original unit": units,
    "Pooled mean": feature_mean,
    "Pooled sample SD": feature_scale,
})
pn.Column( "## Center and Scaling of the Data", pn.pane.DataFrame(feature_info.round(4)))


Column
    [0] Markdown(str)
    [1] DataFrame(DataFrame)

In [7]:
profile_limit = 1.05 * max(np.max(np.abs(A)), np.max(np.abs(B)))
feature_plots = []
for j, feature in enumerate(features):
    station_1 = hv.Scatter(
        (dates, A[:, j]), kdims=["Sampling date"], vdims=["Pooled-standardized result"],
        label="Station 1",
    ).opts(color="#0072B2", size=4, tools=["hover"])\
               * hv.Spikes((dates, A[:, j]),"Sampling date", "Pooled-standardized result").opts(color="#0072B2")
    station_2 = hv.Scatter(
        (dates, B[:, j]), kdims=["Sampling date"], vdims=["Pooled-standardized result"],
        label="Station 2",
    ).opts(color="#D55E00", size=4, tools=["hover"])\
               * hv.Spikes((dates, B[:, j]),"Sampling date", "Pooled-standardized result").opts(color="#D55E00")

    zero = hv.HLine(0).opts(color="#777777", line_dash="dashed", line_width=1)
    feature_plots.append((station_1 * station_2 * zero).opts(
        title=f"{feature} ({units[j]})", width=430, height=250,
        legend_position="top_left", show_grid=True,
        ylim=(-profile_limit, profile_limit),
    ))

profile_layout = hv.Layout(feature_plots).cols(2)
display(pn.Column(
    pn.pane.HoloViews(profile_layout, sizing_mode="stretch_width"),
    pn.pane.Markdown("**Figure 1.** Each curve is a pooled-standardized measurement across the eight matched sampling rounds. Similar profiles do not imply identical station behavior; the GSVD will examine their joint directions."),
))


Column
    [0] HoloViews(Layout, sizing_mode='stretch_width')
    [1] Markdown(str)

# 3. Build the Matrix Pair

Each row corresponds to one matched sampling date; each column corresponds to one indicator.  
Thus $A,B\in\mathbb{R}^{8\times 5}$ act on the same five-dimensional **indicator-weight space**.  
The letters $A$ and $B$ here are matrix labels, not claims that either station is a model operator.

For weights $x\in\mathbb{R}^5$, $A x$ is the Station 1 score across sampling dates and $Bx$ is the Station 2 score.  
The common feature centering makes these scores contrasts about the pooled mean;  
the common scaling puts the five indicator coordinates on comparable units.  
The GSVD then compares the norms of these two score vectors.

In [8]:
l_show(
    L(r"A = "), np.round(A, 3),
    L(r"\quad B = "), np.round(B, 3),
)

rank_A = np.linalg.matrix_rank(A)
rank_B = np.linalg.matrix_rank(B)
rank_pair = np.linalg.matrix_rank(np.vstack((A, B)))
display(diagnostic_table({
    "shape of A": A.shape,
    "shape of B": B.shape,
    "rank(A)": rank_A,
    "rank(B)": rank_B,
    "rank([A; B])": rank_pair,
}, title="Pair dimensions and rank"))


<IPython.core.display.Latex object>

Column(sizing_mode='fixed')
    [0] Markdown(str)
    [1] HoloViews(Table, height=192, width=600)

The stacked pair has full column rank, so no nonzero indicator-weight vector is invisible to both stations.  
Each station matrix also has full column rank for this selected feature set.  
The data are centered, so the eight row scores describe deviations about the pooled feature means rather than absolute concentrations.

# 4. Compute the GSVD with LAPACK

We use the compact factor convention of [**GSVD_computations.ipynb**](GSVD_computations.ipynb):

$\qquad
A X=U C,\qquad B X=V S,\qquad C^T C+S^T S=I.
$

For this full column rank pair, the columns of $U$ and $V$ are orthonormal and $C,S$ are diagonal in the generalized right-vector coordinates. If $x_i$ is column $i$ of $X$, then

$\qquad\displaystyle{
\frac{\lVert A x_i\rVert_2}{\lVert B x_i\rVert_2}=\frac{c_i}{s_i}=\gamma_i.
}$

The numerical factorization is performed by Julia's LAPACK-backed `svd(A, B)`, not by forming normal equations.

In [9]:
if rank_pair < A.shape[1]:
    raise ValueError("The stacked pair must have full column rank for this factor convention")

lapack_result = Main.lapack_gsvd_factors(A, B)
factors = {
    "U": np.asarray(lapack_result.U, dtype=float),
    "V": np.asarray(lapack_result.V, dtype=float),
    "C": np.asarray(lapack_result.C, dtype=float),
    "S": np.asarray(lapack_result.S, dtype=float),
    "X": np.asarray(lapack_result.X, dtype=float),
    "c": np.asarray(lapack_result.c, dtype=float),
    "s": np.asarray(lapack_result.s, dtype=float),
}

c = factors["c"]
s = factors["s"]
gamma = np.divide(c, s, out=np.full_like(c, np.inf), where=~np.isclose(s, 0.0))
order = np.argsort(gamma)[::-1]

mode_table = pd.DataFrame({
    "Mode (ordered by gamma)": np.arange(1, len(order) + 1),
    "LAPACK component": order + 1,
    "c": c[order],
    "s": s[order],
    "gamma = c/s": gamma[order],
    "theta (degrees)": np.degrees(np.arctan2(s[order], c[order])),
})
pn.Column("## GSVD Results", pn.pane.DataFrame(mode_table.round(5)))


Column
    [0] Markdown(str)
    [1] DataFrame(DataFrame)

In [10]:
C, S, X = factors["C"], factors["S"], factors["X"]
U, V = factors["U"], factors["V"]
relative_A = np.linalg.norm(A @ X - U @ C) / np.linalg.norm(A)
relative_B = np.linalg.norm(B @ X - V @ S) / np.linalg.norm(B)
cs_error = np.linalg.norm(C.T @ C + S.T @ S - np.eye(A.shape[1]))
U_error = np.linalg.norm(U.T @ U - np.eye(U.shape[1]))
V_error = np.linalg.norm(V.T @ V - np.eye(V.shape[1]))

display(diagnostic_table({
    "relative reconstruction error, A": relative_A,
    "relative reconstruction error, B": relative_B,
    "||C^T C + S^T S - I||": cs_error,
    "||U^T U - I||": U_error,
    "||V^T V - I||": V_error,
    "full-column-rank stacked pair": rank_pair == A.shape[1],
}, title="LAPACK factorization checks"))


Column(sizing_mode='fixed')
    [0] Markdown(str)
    [1] HoloViews(Table, height=224, width=600)

## 4.1 Check the Ratio Interpretation

The generalized singular values should agree with the relative actions along the computed right directions.  
We verify this directly using the original matrix pair.  
This check is useful because it ties the reported scalar values to the operator action on actual vectors,  
rather than only to the factorization's internal outputs.

In [11]:
right_actions_A = np.linalg.norm(A @ X, axis=0)
right_actions_B = np.linalg.norm(B @ X, axis=0)
direct_gamma = np.divide(
    right_actions_A, right_actions_B,
    out=np.full_like(right_actions_A, np.inf),
    where=right_actions_B > 100 * np.finfo(float).eps,
)

ratio_check = pd.DataFrame({
    "LAPACK component": np.arange(1, len(c) + 1),
    "||A x_i|| / ||B x_i||": direct_gamma,
    "c_i / s_i": gamma,
})
display(pn.pane.DataFrame(ratio_check.round(6)))


DataFrame(DataFrame)

# 5. Read the Cosine-Sine Pairs

Each pair $(c_i,s_i)$ lies on the unit quarter-circle. A point nearer the horizontal axis has larger relative Station 1 action;  
a point nearer the vertical axis has larger relative Station 2 action.  
The ratio $\gamma_i=c_i/s_i$ is the slope reciprocal in this plot's coordinates.

The plot describes **modes of the selected, standardized, mean-centered matrices**.  
It does not identify an individual chemical as responsible: each right vector is a linear combination of all five indicators.

In [21]:
theta = np.linspace(0, np.pi / 2, 200)
unit_arc = hv.Curve((np.cos(theta), np.sin(theta)), "c", "s").opts(
    color="#555555", line_dash="dashed", line_width=1.5,
)
segments = hv.Path([[(0, 0), (float(ci), float(si))] for ci, si in zip(c, s)]).opts(
    color="#9AA0A6", line_width=1,
)
pair_points = hv.Points(
    (c, s, np.arange(1, len(c) + 1)), kdims=["c", "s"], vdims=["mode"],
).opts(
    color="#0072B2", size=8, tools=["hover"],
    hover_tooltips=[("Mode", "@mode"), ("c", "@c{0.000}"), ("s", "@s{0.000}")],
)
pair_plot = (unit_arc * segments * pair_points).opts(
    width=520, height=430, title="Relative GSVD actions by mode",
    xlabel="Station 1 action: c", ylabel="Station 2 action: s",
    xlim=(-0.04, 1.04), ylim=(-0.04, 1.04), aspect="equal", show_grid=True,
)
display(pn.Column(
    pn.pane.HoloViews(pair_plot),
    pn.pane.LaTeX(r"$\textbf{Figure 2.}$ Each point is one GSVD mode; the line from the origin shows its $(c_i,s_i)$ direction. The dashed arc is $c^2+s^2=1$. Mode numbers in the hover tool refer to LAPACK's ordering."),
))


Column
    [0] HoloViews(Overlay, height=430, sizing_mode='fixed', width=520)
    [1] LaTeX(str, renderer='katex')

# 6. What do the Extreme Modes Combine?

The largest $\gamma_i$ maximizes the Station 1-to-Station 2 action ratio among the GSVD directions; the smallest minimizes it.  
To see what each direction means in this example, inspect the corresponding right-vector coefficients.  
Their signs are arbitrary: multiplying a GSVD right vector by $-1$ changes neither its ratio nor its interpretation.

In [13]:
largest_index = int(order[0])
smallest_index = int(order[-1])

def unit_direction(index):
    vector = X[:, index]
    return vector / np.linalg.norm(vector)

largest_direction = unit_direction(largest_index)
smallest_direction = unit_direction(smallest_index)
weight_table = pd.DataFrame({
    "Indicator weight": features,
    "Largest gamma mode": largest_direction,
    "Smallest gamma mode": smallest_direction,
})
display(pn.pane.DataFrame(weight_table.round(4)))

smallest_scores_A = A @ X[:, smallest_index]
smallest_scores_B = B @ X[:, smallest_index]


DataFrame(DataFrame)

In [20]:
weight_plots = []
for label, index, vector in [
    ("Largest gamma", largest_index, largest_direction),
    ("Smallest gamma", smallest_index, smallest_direction),
]:
    weight_plots.append(hv.Bars(
        (features, vector), kdims=["Indicator"], vdims=["Unit-norm coefficient"],
    ).opts(
        width=470, height=300, title=f"{label}: gamma={gamma[index]:.3f}",
        color="#009E73", tools=["hover"], xrotation=25, show_grid=True,
    ))

display(pn.Column(
    pn.pane.HoloViews(hv.Layout(weight_plots).cols(2), sizing_mode="stretch_width"),
    pn.pane.LaTeX("**Figure 3.** Coefficients of Euclidean-unit versions of the two extreme right directions. These normalized coefficients are for easier visual comparison; the GSVD's original column scaling is retained when computing $c_i$ and $s_i$. The mode sign may flip without changing the decomposition."),
))


Column
    [0] HoloViews(Layout, sizing_mode='stretch_width')
    [1] LaTeX(str, renderer='katex')

In [19]:
def score_curve(values, label, color):
    return hv.Scatter(
        (dates, values), kdims=["Sampling date"], vdims=["GSVD score"], label=label,
    ).opts(color=color, size=4, tools=["hover"])\
    * hv.Spikes((dates, values),"Sampling date", "GSVD score").opts(color=color)

extreme_score_plots = []
for title, index in [("Largest gamma", largest_index), ("Smallest gamma", smallest_index)]:
    plot = (
        score_curve(A @ X[:, index], "Station 1", "#0072B2")
        * score_curve(B @ X[:, index], "Station 2", "#D55E00")
        * hv.HLine(0).opts(color="#777777", line_dash="dashed", line_width=1)
    ).opts(
        width=470, height=300, title=f"{title} (gamma={gamma[index]:.3f})",
        xlabel="Sampling date", ylabel="Score", legend_position="bottom_left", show_grid=True,
    )
    extreme_score_plots.append(plot)

display(pn.Column(
    pn.pane.HoloViews(hv.Layout(extreme_score_plots).cols(2), sizing_mode="stretch_width"),
    pn.pane.LaTeX(r"$\textbf{Figure 4.}$ The two station score series are the matrix actions $Ax_i$ and $Bx_i$. Their Euclidean norm ratio is the corresponding generalized singular value. The eight observations are repeated seasonal rounds, not independent replicates for a statistical test."),
))


Column
    [0] HoloViews(Layout, sizing_mode='stretch_width')
    [1] LaTeX(str, renderer='katex')

# 7. An Independent Special-case Check

Both station matrices have full column rank, so $B^T B$ is positive definite.  
That permits the symmetric-definite generalized eigenproblem

$\qquad
A^T A z=\lambda B^T B z,
\qquad \lambda_i=\gamma_i^2.
$

This is a useful independent check and links this application to [**GEP_computation.ipynb**](GEP_computation.ipynb).  
It is **not** the GSVD algorithm used above: forming Gram matrices squares condition numbers.  
We use it only to verify the values on this small, well-conditioned example.

In [16]:
gram_eigenvalues = eigh(A.T @ A, B.T @ B, eigvals_only=True)
gsvd_squared = np.sort(gamma**2)
gep_squared = np.sort(gram_eigenvalues)

display(diagnostic_table({
    "||sort(gamma^2) - GEP eigenvalues||": np.linalg.norm(gsvd_squared - gep_squared),
    "condition number of A": np.linalg.cond(A),
    "condition number of B": np.linalg.cond(B),
    "smallest eigenvalue of B^T B": np.linalg.eigvalsh(B.T @ B).min(),
}, title="GEP cross-check"))


Column(sizing_mode='fixed')
    [0] Markdown(str)
    [1] HoloViews(Table, height=160, width=600)

## 7.1 Sensitivity to Preprocessing and Feature Selection

The GSVD compares the matrices it is given, so centering, scaling, and feature selection define the question being asked. The following check changes one choice at a time. It reports the two largest generalized singular values and the absolute cosine between the leading right direction and the baseline direction. Coefficients are converted back to raw-variable coordinates before comparing; for a feature-omission case, the baseline direction is restricted to the retained features.

The original-unit case is included only to show how incompatible measurement units can dominate the result; it is not a recommended comparison. Separate centering removes station mean offsets and therefore asks about within-station scatter instead of pooled-centered second-moment energy.

These are descriptive sensitivity checks, not uncertainty estimates. With only eight matched sampling rounds, they cannot establish that a mode is stable in the wider environmental process.

In [17]:
def sensitivity_result(label, columns, centering, scaling):
    columns = np.asarray(columns, dtype=int)
    raw_a = A_raw[:, columns]
    raw_b = B_raw[:, columns]
    pooled = np.vstack((raw_a, raw_b))

    if scaling == "pooled SD":
        scale = pooled.std(axis=0, ddof=1)
    else:
        scale = np.ones(len(columns))

    if centering == "pooled mean":
        mean_a = mean_b = pooled.mean(axis=0)
    elif centering == "station means":
        mean_a = raw_a.mean(axis=0)
        mean_b = raw_b.mean(axis=0)
    else:
        mean_a = mean_b = np.zeros(len(columns))

    a = (raw_a - mean_a) / scale
    b = (raw_b - mean_b) / scale
    result = Main.lapack_gsvd_factors(a, b)
    c_values = np.asarray(result.c, dtype=float)
    s_values = np.asarray(result.s, dtype=float)
    ratios = np.divide(
        c_values, s_values, out=np.full_like(c_values, np.inf),
        where=~np.isclose(s_values, 0.0),
    )
    order_local = np.argsort(ratios)[::-1]
    right_basis = np.asarray(result.X, dtype=float)

    # Convert each right direction from preprocessed to raw feature coordinates.
    direction = np.zeros(len(features))
    direction[columns] = right_basis[:, order_local[0]] / scale
    direction /= np.linalg.norm(direction)
    baseline_on_shared_features = baseline_raw_direction[columns]
    baseline_norm = np.linalg.norm(baseline_on_shared_features)
    alignment = (
        abs(np.dot(direction[columns], baseline_on_shared_features / baseline_norm))
        if baseline_norm > 0 else np.nan
    )

    return {
        "Scenario": label,
        "Largest gamma": ratios[order_local[0]],
        "Second-largest gamma": ratios[order_local[1]],
        "Absolute cosine with baseline leading direction": alignment,
    }

baseline_raw_direction = X[:, largest_index] / feature_scale
baseline_raw_direction /= np.linalg.norm(baseline_raw_direction)
all_columns = np.arange(len(features))
sensitivity_rows = [
    sensitivity_result("Baseline: pooled mean, pooled SD", all_columns, "pooled mean", "pooled SD"),
    sensitivity_result("Center each station, pooled SD", all_columns, "station means", "pooled SD"),
    sensitivity_result("No centering, pooled SD", all_columns, "none", "pooled SD"),
    sensitivity_result("Pooled mean, original units", all_columns, "pooled mean", "original units"),
]
sensitivity_rows.extend(
    sensitivity_result(
        f"Omit {features[j]}",
        np.delete(all_columns, j),
        "pooled mean",
        "pooled SD",
    )
    for j in range(len(features))
)
sensitivity_table = pd.DataFrame(sensitivity_rows)
display(pn.pane.DataFrame(sensitivity_table.round(4), sizing_mode="stretch_width"))

DataFrame(DataFrame, sizing_mode='stretch_width')

# 8. Conclusions and Limitations

The GSVD supplies a shared basis for comparing how two rectangular data operators act on the same indicator-weight space.  
In this case, the factors summarize relative signal energy about the pooled mean after a declared common scaling. The objective combines within-station temporal scatter with station mean offsets and does not use date-by-date pairing.

Three distinctions matter when reading the result:

1. **A mode is a contrast, not a single analyte.** Its right vector combines standardized indicator columns; coefficient signs and magnitudes should be interpreted together.
2. **A large $\gamma_i$ is relative to this preprocessing and this station pair.** It is not a regulatory threshold, a risk score, or a water-quality ranking.
3. **Eight matched dates do not support broad environmental conclusions.** Seasonal variability, sampling design, measurement uncertainty, and the selected features all affect the matrices. The example demonstrates linear algebra on real records; it is not an inferential study. With a larger representative sample, uncertainty analysis would still be needed before treating a mode as a repeatable station difference.

For a numerical methods discussion, compare the direct orthogonal GSVD with the Gram-matrix GEP cross-check. Section 7.1 shows how centering, scaling, and feature selection change the leading modes.

## Data References

- [NJDEP Lake Monitoring Program](https://dep.nj.gov/wms/bfbm/lake-monitoring/).
- [WQP Chapin Pond Station 1 profile](https://www.waterqualitydata.us/provider/STORET/NJDEP_BFBM/NJDEP_BFBM-NJLM-0188-1/).
- [WQP Chapin Pond Station 2 profile](https://www.waterqualitydata.us/provider/STORET/NJDEP_BFBM/NJDEP_BFBM-NJLM-0188-2/).
- [Water Quality Portal web-services documentation](https://www.waterqualitydata.us/webservices_documentation/).
- [NJDEP Lakes Quality Assurance Project Plan, 2022--23](https://www.nj.gov/dep/wms/bfbm/docs/Lakes%20QA%20Plan%202022-23%20with%20071522%20amendments.pdf).

To refresh the embedded snapshot, query the WQP `Result/search` service for both identifiers and retain routine samples with the exact characteristic names, units, and fractions listed above. Review the returned qualifiers and sample fractions before updating the arrays. The notebook intentionally does not fetch live data during execution.